# AUtoMLy final-test public replay (October 19, 2026)

Recomputes the reported aggregate metrics from 600 **de-identified final-rating records**. It makes no model calls. It does not contain benchmark questions or keys, source passages or page locators, model answer text, raw judge requests/responses, or the signed review spreadsheet. The private repository retains the complete audit record. This replay checks the arithmetic of the final ratings; it cannot independently reassess factual correctness or source support without the withheld content.

## 1. Load and validate the scrubbed data bundle

In [1]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'README.md').exists() or not (ROOT / 'notebook').exists(): ROOT = ROOT.parent
B = ROOT / 'notebook/evidence_final_test_20261019'
sys.path.insert(0, str(B))
from replay_final_test_public import load_rows, replay
rows = load_rows()
manifest = json.loads((B / 'manifest.json').read_text())
assert len(rows) == manifest['records'] == 600
assert len({r['pair_id'] for r in rows}) == manifest['paired_items'] == 150
forbidden = {'question', 'reference_answer', 'evidence_quotes', 'passage_text', 'output_text', 'source_id', 'pdf_page'}
assert not (forbidden & set(rows[0]))
print('Loaded 600 de-identified rating rows; 150 paired question IDs; excluded raw text and source locators.')

Loaded 600 de-identified rating rows; 150 paired question IDs; excluded raw text and source locators.


## 2. Recompute the saved final-test aggregates

In [2]:
stored = json.loads((B / 'final_results_public.json').read_text())
fresh = replay()
assert fresh == stored, 'Recomputed public metrics differ from the saved aggregate.'
print('All aggregate fields reproduced exactly from the scrubbed per-output ratings.')

All aggregate fields reproduced exactly from the scrubbed per-output ratings.


## 3. Headline results

In [3]:
P = fresh['primary_answerable_main']
print('Answerable strict pass: base {}/{} ({:.1f}%), adapter {}/{} ({:.1f}%), difference {:+.1f} points.'.format(P['base']['k'],P['base']['n'],P['base']['pct'],P['adapter']['k'],P['adapter']['n'],P['adapter']['pct'],P['diff_points']))
for label, key in [('Unsupported claims','unsupported_claim_main'), ('Correct abstention','correct_abstention_main'), ('Correct citations','citation_correctness_main')]:
    vals = fresh[key]
    print(label + ':', ', '.join('{} {}/{} ({:.1f}%)'.format(c,vals[c]['k'],vals[c]['n'],vals[c]['pct']) for c in ('base','adapter')))
print('PM thresholds met:', fresh['pm_thresholds'])
print('Judge agreement counts:', json.dumps(fresh['judge_agreement'], sort_keys=True))
print('Human review: main outputs reviewed =', fresh['andy_review']['main_outputs_reviewed'], '; cut disagreements reviewed =', fresh['andy_review']['cut_disagreements_reviewed'], '; flags =', fresh['andy_review']['flags'])
assert (P['base']['k'], P['adapter']['k']) == (114,111)
assert all(v is False for v in fresh['pm_thresholds'].values())

Answerable strict pass: base 114/120 (95.0%), adapter 111/120 (92.5%), difference -2.5 points.
Unsupported claims: base 6/150 (4.0%), adapter 25/150 (16.7%)
Correct abstention: base 24/30 (80.0%), adapter 2/30 (6.7%)
Correct citations: base 2/120 (1.7%), adapter 11/120 (9.2%)
PM thresholds met: {'adapter_at_least_10pts_over_base': False, 'citation_correct_at_least_95pct_adapter': False, 'unsupported_below_2pct_adapter': False, 'correct_abstention_at_least_90pct_adapter': False}
Judge agreement counts: {"cut60": {"A_strict_pass": 181, "B_strict_pass": 180, "andy_chose_A": 12, "andy_chose_B": 5, "judges_differ_on_any_field": 17, "rows": 300, "strict_pass_agree": 291}, "main": {"A_strict_pass": 253, "B_strict_pass": 255, "andy_chose_A": 5, "andy_chose_B": 4, "judges_differ_on_any_field": 9, "rows": 300, "strict_pass_agree": 292}}
Human review: main outputs reviewed = 300 ; cut disagreements reviewed = 17 ; flags = 0
